# FoodSeg Upload Demo

This notebook reuses artifacts exported by `foodseg-hybrid-30min-modal.ipynb` so you can run inference without training again.

Model loading is intentionally simple:
- If `outputs/**/best_model.zip` exists, the notebook auto-picks the newest one.
- If not, set `BEST_MODEL_ZIP` to the path of your zip file.

This avoids `FileUpload` state issues in VS Code notebooks.

Run the notebook top to bottom.

In [ ]:
# Uncomment in a fresh runtime if torch/torchvision are missing.
# %pip install -q -U torch torchvision
%pip install -q -U transformers pillow matplotlib ipywidgets safetensors gradio

In [ ]:
import io
import json
import shutil
import zipfile
from contextlib import nullcontext
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import gradio as gr
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

ROOT = Path.cwd()
WORK_DIR = ROOT / "demo_upload_workspace"
ARTIFACT_DIR = WORK_DIR / "uploaded_files"
MODEL_DIR = WORK_DIR / "best_model"
IMAGE_DIR = WORK_DIR / "images"

for path in [WORK_DIR, ARTIFACT_DIR, MODEL_DIR, IMAGE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

WEIGHT_FILES = (
    "model.safetensors",
    "pytorch_model.bin",
    "tf_model.h5",
    "flax_model.msgpack",
    "model.safetensors.index.json",
    "pytorch_model.bin.index.json",
)

@dataclass
class LoadedArtifacts:
    model_dir: Path | None = None
    processor: Any = None
    model: Any = None
    id2label: dict[int, str] = field(default_factory=dict)
    label2id: dict[str, int] = field(default_factory=dict)
    metrics: dict[str, Any] = field(default_factory=dict)
    train_config: dict[str, Any] = field(default_factory=dict)
    palette: np.ndarray | None = None

MODEL_BUNDLE = LoadedArtifacts()

print({
    "device": DEVICE,
    "bf16": BF16,
    "workspace": str(WORK_DIR),
})

In [ ]:
BEST_MODEL_ZIP = ROOT / "best_model.zip"
# Example override:
# BEST_MODEL_ZIP = ROOT / "outputs" / "foodseg_30min_segformer_b2_20260425_034421" / "best_model.zip"

root_zip = ROOT / "best_model.zip"
auto_found_zips = []
if root_zip.exists():
    auto_found_zips.append(root_zip)
auto_found_zips.extend(
    sorted(
        [path for path in ROOT.glob("outputs/**/best_model.zip") if path != root_zip],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
)
display(Markdown("## 1. Select `best_model.zip`"))
print("Candidate zip files:")
for path in auto_found_zips[:10]:
    print("-", path)
if BEST_MODEL_ZIP is not None:
    print(f"\nUsing selected zip: {BEST_MODEL_ZIP}")
elif auto_found_zips:
    BEST_MODEL_ZIP = auto_found_zips[0]
    print(f"\nUsing auto-detected zip: {BEST_MODEL_ZIP}")
else:
    print("\nNo best_model.zip found. Put best_model.zip in the repo root or set BEST_MODEL_ZIP manually.")

In [ ]:
def normalize_uploaded_files(value) -> list[dict[str, bytes]]:
    if not value:
        return []
    items = value.values() if isinstance(value, dict) else value
    normalized = []
    for item in items:
        name = item["name"] if isinstance(item, dict) else item.name
        content = item["content"] if isinstance(item, dict) else item.content
        if isinstance(content, memoryview):
            content = content.tobytes()
        elif hasattr(content, "tobytes"):
            content = content.tobytes()
        normalized.append({"name": Path(name).name, "content": bytes(content)})
    return normalized


def reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def read_json_if_exists(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def build_palette(num_labels: int) -> np.ndarray:
    rng = np.random.default_rng(42)
    palette = rng.integers(0, 255, size=(max(num_labels, 1), 3), dtype=np.uint8)
    palette[0] = np.array([0, 0, 0], dtype=np.uint8)
    return palette


def find_model_root(base_dir: Path) -> Path:
    candidates = [base_dir] + sorted([path for path in base_dir.rglob("*") if path.is_dir()])
    for candidate in candidates:
        has_required = (candidate / "config.json").exists() and (candidate / "preprocessor_config.json").exists()
        has_weights = any((candidate / name).exists() for name in WEIGHT_FILES)
        if has_required and has_weights:
            return candidate
    raise FileNotFoundError("Could not find a valid model directory inside best_model.zip.")


def load_model_from_zip(zip_path_like) -> Path:
    if zip_path_like is None:
        raise ValueError("No best_model.zip selected. Set BEST_MODEL_ZIP or put the zip under outputs/**/best_model.zip.")

    zip_path = Path(zip_path_like).expanduser()
    if not zip_path.is_absolute():
        zip_path = (ROOT / zip_path).resolve()
    if not zip_path.exists():
        raise FileNotFoundError(f"best_model.zip not found: {zip_path}")
    if zip_path.name != "best_model.zip":
        raise ValueError(f"Expected a file named best_model.zip, got: {zip_path.name}")

    reset_dir(ARTIFACT_DIR)
    reset_dir(MODEL_DIR)

    copied_zip = ARTIFACT_DIR / "best_model.zip"
    shutil.copy2(zip_path, copied_zip)
    with zipfile.ZipFile(copied_zip) as zf:
        zf.extractall(MODEL_DIR)

    model_root = find_model_root(MODEL_DIR)
    processor = SegformerImageProcessor.from_pretrained(model_root)
    model = SegformerForSemanticSegmentation.from_pretrained(model_root).to(DEVICE).eval()

    raw_id2label = read_json_if_exists(model_root / "id2label.json")
    if raw_id2label:
        id2label = {int(k): v for k, v in raw_id2label.items()}
    else:
        id2label = {int(k): v for k, v in getattr(model.config, "id2label", {}).items()}

    raw_label2id = read_json_if_exists(model_root / "label2id.json")
    label2id = {str(k): int(v) for k, v in raw_label2id.items()} if raw_label2id else {
        str(v): int(k) for k, v in id2label.items()
    }

    metrics = read_json_if_exists(model_root / "eval_metrics.json")
    train_config = read_json_if_exists(model_root / "train_config.json")

    MODEL_BUNDLE.model_dir = model_root
    MODEL_BUNDLE.processor = processor
    MODEL_BUNDLE.model = model
    MODEL_BUNDLE.id2label = id2label
    MODEL_BUNDLE.label2id = label2id
    MODEL_BUNDLE.metrics = metrics
    MODEL_BUNDLE.train_config = train_config
    MODEL_BUNDLE.palette = build_palette(getattr(model.config, "num_labels", len(id2label) or 1))

    display(Markdown(f"**Loaded model:** `{model_root}`"))
    print("zip:", zip_path)
    print("files:", sorted(path.name for path in model_root.iterdir()))
    if metrics:
        display(Markdown("**Eval metrics**"))
        print(json.dumps(metrics, indent=2))
    if train_config:
        display(Markdown("**Train config**"))
        keep_keys = ["model_name", "image_size", "num_labels", "ignore_index"]
        compact = {key: train_config[key] for key in keep_keys if key in train_config}
        print(json.dumps(compact, indent=2))

    return model_root

In [ ]:
loaded_model_dir = load_model_from_zip(BEST_MODEL_ZIP)

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
TEST_IMAGE_PATHS = sorted(
    [path for path in ROOT.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS],
    key=lambda p: p.name.lower(),
)

display(Markdown("## 2. Auto-detect test images in repo root"))
print("Detected images:")
for path in TEST_IMAGE_PATHS:
    print("-", path.name)
if not TEST_IMAGE_PATHS:
    print("No .jpg/.jpeg/.png images found in the repo root.")

In [ ]:
def require_loaded_model() -> None:
    if MODEL_BUNDLE.model is None or MODEL_BUNDLE.processor is None:
        raise RuntimeError("Load model artifacts first.")


def mask_to_rgb(mask: np.ndarray) -> np.ndarray:
    require_loaded_model()
    safe = np.where(
        (mask >= 0) & (mask < len(MODEL_BUNDLE.palette)),
        mask,
        0,
    ).astype(np.int64)
    return MODEL_BUNDLE.palette[safe]


def overlay_mask(image: Image.Image, mask: np.ndarray, alpha: float = 0.48) -> Image.Image:
    image_np = np.array(image.convert("RGB"))
    color = mask_to_rgb(mask)
    foreground = mask > 0
    blended = image_np.copy()
    blended[foreground] = (image_np[foreground] * (1 - alpha) + color[foreground] * alpha).astype(np.uint8)
    return Image.fromarray(blended)


def predict_food_mask(image: Image.Image) -> np.ndarray:
    require_loaded_model()
    image = image.convert("RGB")
    inputs = MODEL_BUNDLE.processor(images=image, return_tensors="pt").to(DEVICE)
    autocast_context = (
        torch.autocast(device_type="cuda", dtype=torch.bfloat16)
        if DEVICE == "cuda" and BF16
        else nullcontext()
    )
    with torch.inference_mode(), autocast_context:
        logits = MODEL_BUNDLE.model(**inputs).logits
        logits = F.interpolate(logits, size=image.size[::-1], mode="bilinear", align_corners=False)
        pred = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
    return pred


def summarize_mask(mask: np.ndarray, top_k: int = 12) -> list[dict[str, float]]:
    ids, counts = np.unique(mask, return_counts=True)
    total = int(counts.sum()) if len(counts) else 0
    rows = []
    for idx, count in sorted(zip(ids, counts), key=lambda item: item[1], reverse=True):
        idx = int(idx)
        if idx == 0 or idx not in MODEL_BUNDLE.id2label:
            continue
        rows.append({
            "label": MODEL_BUNDLE.id2label[idx],
            "area_pct": round(100.0 * int(count) / max(total, 1), 2),
        })
        if len(rows) >= top_k:
            break
    return rows


def load_root_images(image_paths: list[Path]) -> list[tuple[str, Image.Image]]:
    images = []
    reset_dir(IMAGE_DIR)
    for path in image_paths:
        copied_path = IMAGE_DIR / path.name
        shutil.copy2(path, copied_path)
        images.append((path.name, Image.open(path).convert("RGB")))
    return images


def show_predictions(images: list[tuple[str, Image.Image]]) -> dict[str, list[dict[str, float]]]:
    if not images:
        raise ValueError("No .jpg/.jpeg/.png images found in the repo root.")

    summaries = {}
    fig, axes = plt.subplots(len(images), 3, figsize=(16, 5 * len(images)))
    if len(images) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, (name, image) in enumerate(images):
        pred = predict_food_mask(image)
        summary = summarize_mask(pred)
        top_labels = ", ".join(item["label"] for item in summary[:5]) or "background only"

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"{name} | input")
        axes[row, 1].imshow(mask_to_rgb(pred))
        axes[row, 1].set_title("predicted mask")
        axes[row, 2].imshow(overlay_mask(image, pred))
        axes[row, 2].set_title(top_labels)

        for col in range(3):
            axes[row, col].axis("off")

        summaries[name] = summary

    plt.tight_layout()
    return summaries

In [ ]:
root_images = load_root_images(TEST_IMAGE_PATHS)
if not root_images:
    raise ValueError("No .jpg/.jpeg/.png images found in the repo root.")

summaries = show_predictions(root_images)
for name, rows in summaries.items():
    display(Markdown(f"### {name}"))
    print(json.dumps(rows, indent=2))

## 3. Optional Gradio demo

Run the next cell after the model has loaded. It reuses the loaded model in memory.

In [ ]:
def gradio_predict(image: Image.Image):
    require_loaded_model()
    if image is None:
        return None, None, []
    mask = predict_food_mask(image)
    overlay = overlay_mask(image, mask)
    return overlay, Image.fromarray(mask_to_rgb(mask)), summarize_mask(mask)


demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil", label="Food image"),
    outputs=[
        gr.Image(type="pil", label="Overlay"),
        gr.Image(type="pil", label="Mask"),
        gr.JSON(label="Top labels"),
    ],
    title="FoodSeg Upload Demo",
    description="Model is already loaded from best_model.zip. Drop a single image here to test interactively.",
    flagging_mode="never",
)

demo.launch(share=False, debug=False, prevent_thread_lock=True)